### 05.구조화된 출력 파서 StructuredOutputParser page179
- 키-값(Key-Value) JSON 딕셔너리(dict)로 파싱
- ResponseSchema 기반 정의
- 프롬프트 자동 지시사항 주입
- 파이썬 dict 반환
- Pydantic 또는 TypedDict 사용.
- 파싱기호 누락 등의 파싱 에러가 있을 수 있음.

In [ ]:
# !pip --version

pip 26.2.1 from E:\hanwh2609\rag_one\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
# !pip install dotenv
from dotenv import load_dotenv

# .env파일에 설정된 보안정보를 읽기.
load_dotenv()

True

In [ ]:
# !pip install langchain_classic

In [ ]:
# 오류 ModuleNotFoundError
# from output_parsers.output_parsers import ResponseSchema, StructuredOutputParser
# 수정제안
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [7]:
# 사용자의 질문에 대한 답변
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다.",
    ),
]

In [8]:
# 응답 스키마를 기반으로 한 구조화된 출력 파서 초기화
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [9]:
print(output_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"answer": string  // 사용자의 질문에 대한 답변
	"source": string  // 사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다.
}
```


In [10]:
# 출력 형식 지시사항을 파싱합니다.
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    # 사용자의 질문에 최대한 답변하도록 템플릿을 설정합니다.
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    # 입력 변수로 'question'을 사용합니다.
    input_variables=["question"],
    # 부분 변수로 'format_instructions'을 사용합니다.
    partial_variables={"format_instructions": format_instructions},
)

In [11]:
model = ChatOpenAI(temperature=0)  # ChatOpenAI 모델 초기화
chain = prompt | model | output_parser  # 프롬프트, 모델, 출력 파서를 연결

In [12]:
chain.invoke({"question": "아르헨티나의 수도는 어디인가요?"})

{'answer': '아르헨티나의 수도는 부에노스아이레스입니다.',
 'source': 'https://ko.wikipedia.org/wiki/%EC%95%84%EB%A5%B4%EA%B2%A8%ED%8B%80%EB%8B%88%EC%95%84'}

In [13]:
for s in chain.stream({"question": "세종대왕의 업적은 무엇인가요?"}):
    # 스트리밍 출력
    print(s)

{'answer': '세종대왕은 한글을 창제하고 문화를 발전시키는 등 다양한 업적을 가지고 있습니다.', 'source': 'https://ko.wikipedia.org/wiki/%EC%84%B8%EC%A2%85%EB%8C%80%EC%99%95'}


### JSON 형식 출력 파서